This is the cleaning I used to prepare the data for my analysis

LOAD IN LIBRARIES FIRST

In [1]:
import pandas as pd #data manipulation
import numpy as np
import matplotlib.pyplot as plt #visualization
import seaborn as sn #visualization
import requests #api call

#Loading in Incidents Data Using an API Call

In [2]:
#creating a function that loads in the data using an API call
def api_to_dataframe(url):
   #getting the total number of rows in the API
    count_url = f"{url}?$select=count(*)"
    count_response = requests.get(count_url)
    total_count = int(count_response.json()[0]['count']) #this variable indicates the number of records/rows
    print(f"Total rows found: {total_count}")
    #checking if the URL already has query parameters(?) and also setting the limit to the maximum number of records
    if '?' in url:
        url_with_limit = f"{url}&$limit={total_count}"
    else:
        url_with_limit = f"{url}?$limit={total_count}"
    #making the web request to fetch the data and storing the result in a pandas dataframe
    response = requests.get(url_with_limit)
    data = response.json()
    return pd.DataFrame(data)

#Calling in the function and plugging in the api url from data montgomery
incidents_df = api_to_dataframe("https://data.montgomerycountymd.gov/resource/bhju-22kf.json")

# dropping  columns that are not part of the data and checking the first few lines
columns_to_drop = [col for col in incidents_df.columns if col.startswith(':')]
incidents_df = incidents_df.drop(columns=columns_to_drop)

Total rows found: 121640


##Cleaning Incidents Data

In [3]:
#removing capitalization for different verisons of the same categorical values
cols_to_clean = ['light', 'weather', 'surface_condition', 'collision_type', 'road_grade', 'road_condition']
for col in cols_to_clean:
  incidents_df[col] = incidents_df[col].str.lower()

#generalizing categorical values
incidents_df["weather"] = incidents_df["weather"].replace({'raining': 'rain', 'blowing snow': 'snow','foggy':'fog, smog, smoke', 'sleet':'sleet or hail', 'severe crosswinds':'severe winds', 'freezing rain or freezing drizzle':'wintry mix'})
incidents_df["weather"] = incidents_df["weather"].replace({ 'blowing sand, soil, dirt':'adverse weather', 'fog, smog, smoke': 'adverse weather', 'severe winds': 'adverse weather', 'rain':'precipitation', 'wintry mix': 'precipitation', 'snow':'precipitation', 'sleet or hail': 'precipitation', 'other':'unknown'})
incidents_df['light'] = incidents_df['light'].replace({'dark - lighted': 'dark', 'dark - not lighted': 'dark', 'dark - unknown lighting': 'dark', 'dark -- unknown lighting': 'dark', 'dark lights on': 'dark', 'dark no lights': 'dark' })

#coverting date column to datetime object for time-series functionality
if incidents_df.index.name == 'crash_date_time': #check if 'crash_date_time' is already the index and reset if necessary
    incidents_df.reset_index(inplace=True)
incidents_df['crash_date_time'] = pd.to_datetime(incidents_df['crash_date_time'], format='ISO8601')
incidents_df.set_index('crash_date_time', inplace=True)

#making a new copy of the cleaned dataset and storing it under a new name
cleaned_incidents_df = incidents_df.copy()

### Exporting Cleaned Data

In [8]:
#exporting the cleaned_incidents_df to a CSV file
cleaned_incidents_df.to_csv('cleaned_incidents_data.csv', index=False)

#Loading in Drivers Data Using an API Call

In [4]:
#loading in the second dataset and checking the first few lines
drivers_df = api_to_dataframe("https://data.montgomerycountymd.gov/resource/mmzv-x632.json") #using the previously created function
columns_to_drop = [col for col in drivers_df.columns if col.startswith(':')]  #
drivers_df = drivers_df.drop(columns=columns_to_drop)

Total rows found: 214273


##Cleaning Drivers Data

In [5]:
#removing capitalization for different verisons of the same categorical values
cols_to_clean = ['light', 'weather', 'surface_condition', 'collision_type', 'injury_severity', 'vehicle_damage_extent', 'vehicle_movement', 'vehicle_first_impact_location']
for col in cols_to_clean:
  drivers_df[col] = drivers_df[col].str.lower()

#generalizing categorical values
drivers_df["weather"] = drivers_df["weather"].replace({'raining': 'rain', 'blowing snow': 'snow','foggy':'fog, smog, smoke', 'sleet':'sleet or hail', 'severe crosswinds':'severe winds', 'freezing rain or freezing drizzle':'wintry mix'})
drivers_df['light'] = drivers_df['light'].replace({'dark - lighted': 'dark', 'dark - not lighted': 'dark', 'dark - unknown lighting': 'dark', 'dark -- unknown lighting': 'dark', 'dark lights on': 'dark', 'dark no lights': 'dark' })

#coverting date column to datetime object
if drivers_df.index.name == 'crash_date_time': #check if 'crash_date_time' is already the index and reset if necessary
    drivers_df.reset_index(inplace=True)
drivers_df['crash_date_time'] = pd.to_datetime(drivers_df['crash_date_time'], format='ISO8601')
drivers_df.set_index('crash_date_time', inplace=True)

#making a new copy of the cleaned dataset and storing it under a new name
cleaned_drivers_df = drivers_df.copy()

###Exporting Cleaned Data

In [9]:
#exporting the cleaned_drivers_df to a CSV file
cleaned_drivers_df.to_csv('cleaned_drivers_data.csv', index=False)